In [1]:
!pip install xgboost
from xgboost import XGBRegressor

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set the random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
# Load datasets with SMILES column
train_data = pd.read_csv('/kaggle/input/scaffold-descriptors/scaffold_train_descriptors.csv')
validation_data = pd.read_csv('/kaggle/input/scaffold-descriptors/scaffold_validation_descriptors.csv')
test_data = pd.read_csv('/kaggle/input/scaffold-descriptors/scaffold_test_descriptors.csv')

In [4]:
def preprocess_dataset(df, include_smiles=False):
    # Remove unnecessary columns but keep 'weight'
    columns_to_drop = ['Molecule', 'scaffold', 'split']
    df_cleaned = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
    
    # Separate features, target, and weights
    if include_smiles:
        X = df_cleaned.drop(['Affinity', 'weight'], axis=1)
        smiles = df_cleaned['SMILES']
    else:
        X = df_cleaned.drop(['Affinity', 'SMILES', 'weight'], axis=1)
        smiles = df_cleaned['SMILES']
    
    y = df_cleaned['Affinity']
    weights = df_cleaned['weight']
    
    return X, y, smiles, weights

# def preprocess_dataset(df, include_smiles=False):
#     # Drop only the columns we know we don't need
#     columns_to_drop = ['Molecule', 'scaffold', 'split']
#     df_cleaned = df.drop(columns=[c for c in columns_to_drop if c in df.columns],
#                          errors='ignore')
    
#     # Extract SMILES if requested
#     if 'SMILES' in df_cleaned.columns: #include_smiles and 
#         smiles = df_cleaned['SMILES']
#         X = df_cleaned.drop(['Affinity', 'SMILES'], axis=1, errors='ignore')
#     # else:
#     #     smiles = None
#     #     X = df_cleaned.drop(['Affinity'], axis=1, errors='ignore')
    
#     y = df_cleaned['Affinity']
    
#     # Override any existing weight column with all 1’s
#     weights = np.ones(len(df_cleaned), dtype=float)
    
#     return X, y, smiles, weights


In [5]:
# Preprocess datasets
X_train, y_train, smiles_train, weights_train = preprocess_dataset(train_data)
X_val, y_val, smiles_val, weights_val = preprocess_dataset(validation_data)
X_test, y_test, smiles_test, weights_test = preprocess_dataset(test_data)
# print(X_train)
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [6]:
# Hyperparameter tuning configurations
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 125], #[75, 100, 125]
        'max_depth': [20, 30], #[10, 15, 20]
        'min_samples_split': [2, 5, 10], #[2, 5, 10]
        'min_samples_leaf': [1, 2, 4] #[1, 2, 4]
    },
    'Bagging': {
        'n_estimators': [75, 100, 125],
        'max_samples': [0.5, 0.75, 1.0],
        'max_features': [0.5, 0.75, 1.0]
    },
    'Gradient Boosting': {
        'n_estimators': [75, 100, 125],
        'learning_rate': [0.001, 0.01, 0.1],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5, 10]
    }
    
    ,
    'Support Vector Machine': {
        'C': [0.1, 1, 10],
        'epsilon': [0.1, 0.5, 1, 2],
        'kernel': ['rbf'],#, 'linear', 'poly', 'sigmoid'],
        'gamma': ['auto']#[0.1, 1]
    }
    # 'XGBoost': {
    #     'n_estimators': [75, 100, 125],
    #     'learning_rate': [0.001, 0.01, 0.1],
    #     'max_depth': [3, 5, 7],
    #     # XGBoost uses `min_child_weight` instead of `min_samples_split`
    #     'min_child_weight': [1, 3, 5],
    #     # you can add subsample/colsample settings if you like:
    #     # 'subsample': [0.6, 0.8, 1.0],
    #     # 'colsample_bytree': [0.6, 0.8, 1.0]
    # }
}

In [7]:
def tune_hyperparameters(model, param_grid, X_train, y_train, sample_weights=None):
    grid_search = GridSearchCV(
        estimator=model, 
        param_grid=param_grid, 
        cv=2, #5 
        scoring='neg_mean_squared_error', 
        n_jobs=-1,  # Use all available cores
        verbose=2
    )
    if sample_weights is not None:
        grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        grid_search.fit(X_train, y_train)
    return grid_search

In [8]:
def evaluate_and_save_results(model, X_val, y_val, smiles_val, weights_val, X_test, y_test, smiles_test, weights_test, model_name):
    # Predictions for validation dataset
    val_predictions = model.predict(X_val)
    val_results = pd.DataFrame({
        'SMILES': smiles_val,
        'Affinity': y_val,
        'Affinity_Predicted': val_predictions,
        'Weight': weights_val
    })
    val_results['Model'] = model_name
    val_results.to_csv(f'{model_name}_validation_results.csv', index=False)
    
    # Predictions for test dataset
    test_predictions = model.predict(X_test)
    test_results = pd.DataFrame({
        'SMILES': smiles_test,
        'Affinity': y_test,
        'Affinity_Predicted': test_predictions,
        'Weight': weights_test
    })
    test_results['Model'] = model_name
    test_results.to_csv(f'{model_name}_test_results.csv', index=False)
    
    # Compute performance metrics with sample weights
    val_mse = mean_squared_error(y_val, val_predictions, sample_weight=weights_val)
    val_mae = mean_absolute_error(y_val, val_predictions, sample_weight=weights_val)
    val_r2 = r2_score(y_val, val_predictions, sample_weight=weights_val)
    
    test_mse = mean_squared_error(y_test, test_predictions, sample_weight=weights_test)
    test_mae = mean_absolute_error(y_test, test_predictions, sample_weight=weights_test)
    test_r2 = r2_score(y_test, test_predictions, sample_weight=weights_test)
    
    return {
        'Validation_MSE': val_mse,
        'Validation_MAE': val_mae,
        'Validation_R2': val_r2,
        'Test_MSE': test_mse,
        'Test_MAE': test_mae,
        'Test_R2': test_r2
    }

In [9]:
# Models to test
models = {
    'Random Forest': RandomForestRegressor(), #random_state=42
    'Bagging': BaggingRegressor(), #random_state=42
    'Gradient Boosting': GradientBoostingRegressor(), #random_state=42
    'Support Vector Machine': SVR() 
    # 'XGBoost': XGBRegressor(
    #     random_state=42, 
    #     objective='reg:squarederror',  # use squared-error for regression
    #     verbosity=0                  # quiet logging
    # )
}

# Store results
all_model_results = {}

In [10]:
# Perform hyperparameter tuning and evaluation
print("Model Evaluation and Result Saving:")
for name, model in tqdm(models.items(), desc="Model Processing", unit="model"):
    print(f"\nProcessing {name}...")
    
    grid_search = tune_hyperparameters(
        model, 
        param_grids[name], 
        X_train_scaled, 
        y_train,
        weights_train
    )
    
    # Get best model
    best_model = grid_search.best_estimator_
    
    # Print best parameters
    print("Best Parameters:", grid_search.best_params_)
    
    # Evaluate and save results
    model_results = evaluate_and_save_results(
        best_model, 
        X_val_scaled, y_val, smiles_val, weights_val,
        X_test_scaled, y_test, smiles_test, weights_test,
        name
    )
    
    # Store results
    all_model_results[name] = model_results
    
    # Print performance metrics
    print("Model Performance:")
    for metric, value in model_results.items():
        print(f"{metric}: {value}")

Model Evaluation and Result Saving:


Model Processing:   0%|          | 0/4 [00:00<?, ?model/s]


Processing Random Forest...
Fitting 2 folds for each of 36 candidates, totalling 72 fits
Best Parameters: {'max_depth': 30, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Model Performance:
Validation_MSE: 0.12442827246418985
Validation_MAE: 0.26708999921259646
Validation_R2: 0.8412494178507935
Test_MSE: 0.18423000666308495
Test_MAE: 0.31841085627795496
Test_R2: 0.7967783567065518

Processing Bagging...
Fitting 2 folds for each of 27 candidates, totalling 54 fits
Best Parameters: {'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 125}
Model Performance:
Validation_MSE: 0.12851911668704666
Validation_MAE: 0.26958660221356073
Validation_R2: 0.8360301546640677
Test_MSE: 0.18855628787779277
Test_MAE: 0.3262842364179105
Test_R2: 0.7920060940674349

Processing Gradient Boosting...
Fitting 2 folds for each of 81 candidates, totalling 162 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}
Model Performance:
Val

In [11]:
import os

# Ensure a directory for results exists
os.makedirs('best_model_results', exist_ok=True)

# Find the best overall model based on Validation R2
best_model_name = max(all_model_results, key=lambda x: all_model_results[x]['Validation_R2'])
print(f"\nBest Overall Model: {best_model_name}")
print("Best Model Performance:")
for metric, value in all_model_results[best_model_name].items():
    print(f"{metric}: {value}")

# Get the best model from the models dictionary
best_model = models[best_model_name]

# Retrain the best model on the full training dataset
best_model.fit(X_train_scaled, y_train)

# Generate predictions for validation dataset
val_predictions = best_model.predict(X_val_scaled)
validation_results = pd.DataFrame({
    'SMILES': smiles_val,
    'Affinity': y_val,
    'Affinity_Predicted': val_predictions
})
validation_results['Absolute_Error'] = abs(validation_results['Affinity'] - validation_results['Affinity_Predicted'])
validation_results.to_csv('best_model_results/validation_predictions.csv', index=False)

# Generate predictions for test dataset
test_predictions = best_model.predict(X_test_scaled)
test_results = pd.DataFrame({
    'SMILES': smiles_test,
    'Affinity': y_test,
    'Affinity_Predicted': test_predictions
})
test_results['Absolute_Error'] = abs(test_results['Affinity'] - test_results['Affinity_Predicted'])
test_results.to_csv('best_model_results/test_predictions.csv', index=False)

# Save best model performance metrics
performance_metrics = pd.DataFrame.from_dict(
    {best_model_name: all_model_results[best_model_name]}, 
    orient='index'
)
performance_metrics.to_csv('best_model_results/best_model_performance.csv')


Best Overall Model: Gradient Boosting
Best Model Performance:
Validation_MSE: 0.11228484753801567
Validation_MAE: 0.2603074059707605
Validation_R2: 0.8567424865733393
Test_MSE: 0.18630372434430426
Test_MAE: 0.3151267652870357
Test_R2: 0.7944908666144803
